# 📊 UAS Data Science — Week 5 Progress
## Deployment Sederhana: Menyimpan Model & Prediksi Data Baru

**Dataset:** Sales & Marketing Customer Dataset  
**Tujuan Week 5:** Menyimpan model final (.pkl), membangun pipeline prediksi end-to-end, dan menguji model pada data pelanggan baru

---

## 1. Recap Week 4

Pada Week 4 telah dilakukan:
- ✅ Hyperparameter tuning Random Forest dan XGBoost dengan GridSearchCV
- ✅ XGBoost (Tuned) menjadi model dengan performa terbaik
- ✅ Optimasi threshold klasifikasi untuk memaksimalkan Recall kelas Churn

**Rencana Week 5:**
- Melatih ulang model final dengan **seluruh data** (train + test) menggunakan parameter terbaik
- Menyimpan model, scaler, dan label encoders ke file `.pkl`
- Membangun fungsi `predict_churn()` yang menerima data pelanggan baru (raw) dan mengembalikan prediksi
- Menguji pipeline dengan beberapa contoh data baru

---

## 2. Import Library

In [ ]:
import pandas as pd
import numpy as np
import pickle
import joblib

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score

import warnings
warnings.filterwarnings('ignore')

print('✅ Library berhasil diimport')

## 3. Membangun Ulang Pipeline Preprocessing (Reusable Function)

Agar model bisa digunakan untuk data baru (raw, belum diproses), seluruh langkah preprocessing dari Week 2 dibungkus menjadi satu fungsi yang konsisten.

In [ ]:
# Load data mentah
df_raw = pd.read_csv('Sales_-_Marketing_customer_dataset.csv')

# 1. Cleaning: hapus anomali usia
df_raw = df_raw[~((df_raw['age'] < 0) | (df_raw['age'] > 100))].copy()
df_raw.reset_index(drop=True, inplace=True)

# 2. Imputasi missing values — simpan nilai referensi untuk dipakai ulang saat prediksi
median_age = df_raw['age'].median()
median_spent = df_raw['total_spent'].median()
median_sat = df_raw['satisfaction_score'].median()
mode_gender = df_raw['gender'].mode()[0]

df_raw['coupon_code'] = df_raw['coupon_code'].fillna('No Coupon')
df_raw['gender'] = df_raw['gender'].fillna(mode_gender)
df_raw['age'] = df_raw['age'].fillna(median_age)
df_raw['total_spent'] = df_raw['total_spent'].fillna(median_spent)
df_raw['satisfaction_score'] = df_raw['satisfaction_score'].fillna(median_sat)

print('Nilai referensi untuk imputasi (disimpan untuk data baru):')
print(f'  median_age    = {median_age}')
print(f'  median_spent  = {median_spent:.2f}')
print(f'  median_sat    = {median_sat}')
print(f'  mode_gender   = {mode_gender}')

In [ ]:
# 3. Feature Engineering tanggal
df_raw['signup_date'] = pd.to_datetime(df_raw['signup_date'])
df_raw['last_purchase_date'] = pd.to_datetime(df_raw['last_purchase_date'])
reference_date = pd.Timestamp('2025-01-01')

df_raw['customer_tenure_days'] = (reference_date - df_raw['signup_date']).dt.days
df_raw['recency_days'] = (reference_date - df_raw['last_purchase_date']).dt.days
df_raw['signup_year'] = df_raw['signup_date'].dt.year
df_raw['signup_month'] = df_raw['signup_date'].dt.month

# 4. Fitur turunan
df_raw['spend_per_visit'] = df_raw['total_spent'] / (df_raw['total_visits'] + 1)
df_raw['engagement_score'] = (df_raw['email_open_rate'] * 0.5) + (df_raw['email_click_rate'] * 0.5)
df_raw['refund_rate'] = df_raw['refund_requested'] / (df_raw['support_tickets'] + 1)

print('✅ Feature engineering selesai')
print(f'Total kolom: {df_raw.shape[1]}')

In [ ]:
# 5. Drop kolom yang tidak dipakai untuk modeling
drop_cols = ['customer_id', 'signup_date', 'last_purchase_date', 'city']
df_model = df_raw.drop(columns=drop_cols).copy()

# 6. Label Encoding — simpan setiap encoder untuk dipakai ulang saat prediksi data baru
cat_cols = ['gender', 'country', 'acquisition_channel', 'device_type',
            'subscription_type', 'coupon_code', 'payment_method']

label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col].astype(str))
    label_encoders[col] = le

print('✅ Label Encoding selesai, encoder disimpan untuk setiap kolom:')
for col in cat_cols:
    print(f'  {col}: {list(label_encoders[col].classes_)}')

In [ ]:
# 7. Feature Scaling — simpan scaler untuk dipakai ulang
scale_cols = ['age', 'total_visits', 'avg_session_time', 'pages_per_session',
              'email_open_rate', 'email_click_rate', 'total_spent', 'avg_order_value',
              'delivery_delay_days', 'satisfaction_score', 'nps_score',
              'marketing_spend_per_user', 'lifetime_value', 'last_3_month_purchase_freq',
              'customer_tenure_days', 'recency_days', 'spend_per_visit',
              'engagement_score', 'refund_rate']

X = df_model.drop(columns=['churn'])
y = df_model['churn']

scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[scale_cols] = scaler.fit_transform(X[scale_cols])

print('✅ Scaling selesai, scaler disimpan untuk dipakai ulang')
print(f'Shape fitur final: {X_scaled.shape}')

## 4. Melatih Model Final (XGBoost Tuned)

Model final dilatih menggunakan parameter terbaik hasil tuning Week 4, kemudian dilatih ulang dengan seluruh data (train+test) yang sudah di-SMOTE pada bagian train untuk deployment final.

In [ ]:
# Split untuk evaluasi akhir sebelum deployment
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

# Parameter terbaik dari hasil tuning Week 4 (contoh — sesuaikan dengan hasil grid search Anda)
best_params = {
    'n_estimators': 200,
    'max_depth': 5,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'random_state': 42,
    'eval_metric': 'logloss',
    'use_label_encoder': False
}

final_model = XGBClassifier(**best_params)
final_model.fit(X_train_sm, y_train_sm)

print('✅ Model final (XGBoost Tuned) berhasil dilatih')
print(f'Parameter: {best_params}')

In [ ]:
# Evaluasi akhir pada test set
y_pred = final_model.predict(X_test)
print('=== Evaluasi Akhir Model Final ===')
print(f'Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'F1-Score : {f1_score(y_test, y_pred):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Tidak Churn', 'Churn']))

## 5. Menyimpan Model & Preprocessing Object (.pkl)

In [ ]:
# Simpan semua object yang dibutuhkan untuk pipeline prediksi dalam satu file
deployment_artifacts = {
    'model': final_model,
    'scaler': scaler,
    'label_encoders': label_encoders,
    'scale_cols': scale_cols,
    'cat_cols': cat_cols,
    'feature_order': list(X.columns),
    'imputation_values': {
        'median_age': median_age,
        'median_spent': median_spent,
        'median_sat': median_sat,
        'mode_gender': mode_gender
    },
    'reference_date': reference_date
}

with open('churn_model_final.pkl', 'wb') as f:
    pickle.dump(deployment_artifacts, f)

print('✅ Model dan seluruh artefak preprocessing berhasil disimpan ke: churn_model_final.pkl')

import os
size_kb = os.path.getsize('churn_model_final.pkl') / 1024
print(f'Ukuran file: {size_kb:.1f} KB')

## 6. Membangun Fungsi Prediksi End-to-End

Fungsi ini menerima data pelanggan baru dalam format **mentah** (seperti kolom asli CSV) dan langsung mengembalikan prediksi churn beserta probabilitasnya — tanpa perlu preprocessing manual.

In [ ]:
def predict_churn(new_data: pd.DataFrame, artifacts_path='churn_model_final.pkl', threshold=0.42):
    """
    Memprediksi churn untuk data pelanggan baru.

    Parameters
    ----------
    new_data : pd.DataFrame
        Data pelanggan baru dengan kolom mentah (sama seperti dataset asli),
        TANPA kolom 'churn'.
    artifacts_path : str
        Path ke file .pkl hasil training.
    threshold : float
        Threshold probabilitas untuk klasifikasi churn (hasil optimasi Week 4).

    Returns
    -------
    pd.DataFrame dengan kolom tambahan: churn_probability, churn_prediction, risk_label
    """
    with open(artifacts_path, 'rb') as f:
        art = pickle.load(f)

    df = new_data.copy()

    # 1. Cleaning age
    df['age'] = df['age'].clip(lower=0, upper=100)

    # 2. Imputasi missing values pakai nilai referensi training
    iv = art['imputation_values']
    df['coupon_code'] = df['coupon_code'].fillna('No Coupon')
    df['gender'] = df['gender'].fillna(iv['mode_gender'])
    df['age'] = df['age'].fillna(iv['median_age'])
    df['total_spent'] = df['total_spent'].fillna(iv['median_spent'])
    df['satisfaction_score'] = df['satisfaction_score'].fillna(iv['median_sat'])

    # 3. Feature engineering tanggal
    df['signup_date'] = pd.to_datetime(df['signup_date'])
    df['last_purchase_date'] = pd.to_datetime(df['last_purchase_date'])
    ref_date = art['reference_date']
    df['customer_tenure_days'] = (ref_date - df['signup_date']).dt.days
    df['recency_days'] = (ref_date - df['last_purchase_date']).dt.days
    df['signup_year'] = df['signup_date'].dt.year
    df['signup_month'] = df['signup_date'].dt.month

    # 4. Fitur turunan
    df['spend_per_visit'] = df['total_spent'] / (df['total_visits'] + 1)
    df['engagement_score'] = (df['email_open_rate'] * 0.5) + (df['email_click_rate'] * 0.5)
    df['refund_rate'] = df['refund_requested'] / (df['support_tickets'] + 1)

    # 5. Drop kolom yang tidak dipakai
    drop_cols = ['customer_id', 'signup_date', 'last_purchase_date', 'city']
    df_proc = df.drop(columns=[c for c in drop_cols if c in df.columns])

    # 6. Label encoding (pakai encoder hasil training)
    for col in art['cat_cols']:
        le = art['label_encoders'][col]
        df_proc[col] = df_proc[col].astype(str).map(
            lambda x: x if x in le.classes_ else le.classes_[0]
        )
        df_proc[col] = le.transform(df_proc[col])

    # 7. Reorder kolom sesuai urutan training
    df_proc = df_proc[art['feature_order']]

    # 8. Scaling
    df_scaled = df_proc.copy()
    df_scaled[art['scale_cols']] = art['scaler'].transform(df_proc[art['scale_cols']])

    # 9. Prediksi
    proba = art['model'].predict_proba(df_scaled)[:, 1]
    pred = (proba >= threshold).astype(int)

    result = new_data.copy()
    result['churn_probability'] = proba.round(4)
    result['churn_prediction'] = pred
    result['risk_label'] = pd.cut(proba, bins=[0, 0.3, 0.6, 1.0],
                                   labels=['Low Risk', 'Medium Risk', 'High Risk'])
    return result

print('✅ Fungsi predict_churn() berhasil didefinisikan')

## 7. Uji Coba Prediksi pada Data Pelanggan Baru

In [ ]:
# Ambil 5 sampel acak dari data asli sebagai simulasi "data pelanggan baru"
sample_new_customers = pd.read_csv('Sales_-_Marketing_customer_dataset.csv').sample(5, random_state=7)
actual_churn = sample_new_customers['churn'].values
sample_new_customers_input = sample_new_customers.drop(columns=['churn'])

print('Data pelanggan baru (input mentah, tanpa label churn):')
sample_new_customers_input[['customer_id', 'age', 'country', 'subscription_type',
                             'total_spent', 'satisfaction_score', 'nps_score']]

In [ ]:
# Jalankan prediksi end-to-end
hasil_prediksi = predict_churn(sample_new_customers_input)

print('=== Hasil Prediksi ===')
display_cols = ['customer_id', 'country', 'subscription_type', 'satisfaction_score',
                 'churn_probability', 'churn_prediction', 'risk_label']
print(hasil_prediksi[display_cols].to_string(index=False))

print('\n=== Perbandingan dengan Label Aktual (untuk validasi) ===')
for i, (cid, pred, actual) in enumerate(zip(
        hasil_prediksi['customer_id'], hasil_prediksi['churn_prediction'], actual_churn)):
    status = '✅ BENAR' if pred == actual else '❌ SALAH'
    print(f'  Customer {cid}: Prediksi={pred}, Aktual={actual} → {status}')

## 8. Simulasi Input Manual (Single Customer)

Contoh penggunaan untuk satu pelanggan baru yang datanya diinput secara manual — mensimulasikan skenario penggunaan nyata (misalnya dari form aplikasi).

In [ ]:
# Contoh input manual satu pelanggan baru
new_customer = pd.DataFrame([{
    'customer_id': 99999,
    'gender': 'Female',
    'age': 34,
    'country': 'India',
    'city': 'Mumbai',
    'signup_date': '2023-05-10',
    'last_purchase_date': '2024-11-20',
    'acquisition_channel': 'Organic',
    'device_type': 'Mobile',
    'subscription_type': 'Monthly',
    'is_premium_user': 0,
    'total_visits': 12,
    'avg_session_time': 4.5,
    'pages_per_session': 3.2,
    'email_open_rate': 0.25,
    'email_click_rate': 0.05,
    'total_spent': 150.0,
    'avg_order_value': 25.0,
    'discount_used': 1,
    'coupon_code': 'SAVE10',
    'support_tickets': 3,
    'refund_requested': 1,
    'delivery_delay_days': 2,
    'payment_method': 'Credit Card',
    'satisfaction_score': 2.5,
    'nps_score': 4,
    'marketing_spend_per_user': 8.0,
    'lifetime_value': 180.0,
    'last_3_month_purchase_freq': 1
}])

hasil = predict_churn(new_customer)
print('=== Hasil Prediksi untuk Pelanggan Baru ===')
print(f"Customer ID       : {hasil['customer_id'].values[0]}")
print(f"Probabilitas Churn: {hasil['churn_probability'].values[0]*100:.2f}%")
print(f"Prediksi          : {'CHURN' if hasil['churn_prediction'].values[0]==1 else 'TIDAK CHURN'}")
print(f"Tingkat Risiko    : {hasil['risk_label'].values[0]}")

## 9. Memuat Ulang Model dari File (Simulasi Penggunaan di Sistem Lain)

Bagian ini membuktikan bahwa file `.pkl` benar-benar dapat dimuat ulang secara independen (misalnya di aplikasi Flask/Streamlit terpisah) tanpa perlu kode training.

In [ ]:
# Simulasi: "restart" — pura-pura ini sesi Python baru yang hanya punya file .pkl
with open('churn_model_final.pkl', 'rb') as f:
    loaded_artifacts = pickle.load(f)

print('✅ Model berhasil dimuat ulang dari file')
print(f"Tipe model      : {type(loaded_artifacts['model']).__name__}")
print(f"Jumlah fitur     : {len(loaded_artifacts['feature_order'])}")
print(f"Jumlah encoder   : {len(loaded_artifacts['label_encoders'])}")
print('\n✅ Model siap digunakan untuk prediksi tanpa perlu re-training')

## 10. Ringkasan Week 5

### ✅ Yang Telah Dilakukan

| Tahap | Detail |
|-------|--------|
| **Model Final** | XGBoost dengan parameter terbaik hasil tuning Week 4 |
| **Serialisasi** | Model + scaler + label encoders + nilai imputasi disimpan dalam `churn_model_final.pkl` |
| **Fungsi Prediksi** | `predict_churn()` — menerima data mentah, otomatis preprocessing, mengembalikan prediksi + probabilitas + risk label |
| **Uji Coba** | Validasi pada 5 sampel data + 1 contoh input manual |
| **Verifikasi Deployment** | Model dimuat ulang dari file untuk memastikan dapat dipakai independen dari kode training |

### 📦 Output Akhir
- `churn_model_final.pkl` → siap diintegrasikan ke aplikasi (web app, API, atau dashboard)
- Fungsi `predict_churn()` dapat langsung dipanggil dengan data pelanggan baru

### 🗓️ Rencana Week 6 (Final)
- Penyusunan laporan akhir & dokumentasi project secara keseluruhan
- Business recommendation berdasarkan insight dari Week 1–5
- Kesimpulan dan keterbatasan (limitation) project
- Persiapan presentasi/demo